# voiceai — Colab smoke test

Runs the full pipeline on a tiny stand-in model in <10 minutes. Verifies:
  - install works on Colab T4
  - Mimi codec loads + encodes
  - VoiceAILM forward + backward
  - Each training stage runs without NaN

**Cost:** $0 (free T4). **Use:** before any paid GPU run.

In [ ]:
!git clone https://github.com/YOUR_USER/voiceai.git
%cd voiceai
!pip install -q -e '.[train,dev]'

In [ ]:
# 1) Unit tests on CPU (no GPU yet)
!pytest tests/ -q -x -m 'not slow'

In [ ]:
# 2) Verify GPU
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')

In [ ]:
# 3) Run the full smoke harness — same as scripts/smoke_test.py
!python scripts/smoke_test.py

In [ ]:
# 4) Load real Qwen3.5-0.8B + Mimi on T4 — verifies they actually fit
import os
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'

import torch
from voiceai.model.voiceai_lm import VoiceAIConfig, VoiceAILM
from voiceai.model.mimi_utils import load_mimi

cfg = VoiceAIConfig(
    backbone='Qwen/Qwen3.5-0.8B',
    freeze_backbone=True,
    dtype='bfloat16',
    load_in_4bit=False,
)
model = VoiceAILM(cfg).cuda()
mimi = load_mimi(device='cuda', dtype=torch.bfloat16)

print(f'voiceai trainable params: {model.trainable_param_count()/1e6:.1f}M')
print(f'voiceai total params: {model.total_param_count()/1e6:.1f}M')
print(f'mimi loaded, sample rate {mimi.sample_rate}')
print(f'allocated GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# 5) Encode 2 seconds of dummy audio and pass through the model — real shape check
import torch
from voiceai.model.mimi_utils import mimi_encode

audio = torch.randn(1, 1, 24000 * 2, device='cuda', dtype=torch.bfloat16)
codes = mimi_encode(mimi, audio)
print(f'codes shape: {codes.shape}')

with torch.no_grad():
    text_ids = torch.full((1, codes.shape[2]), 0, dtype=torch.long, device='cuda')
    out = model(text_ids=text_ids, user_audio_codes=codes)
print('text_logits:', out['text_logits'].shape)
print('asst_audio_logits:', out['asst_audio_logits'].shape)

If everything above ran without errors, the stack is ready for real training on a paid GPU.